# Notebook 3: Transformer Fine-Tuning (mT5 + IndicBART)
**MarathiMWP Thesis — Chapter 4.4 + Chapter 5**

Fine-tune two seq2seq transformers on the Marathi training set:
- **mT5-small**: Google's multilingual T5, 60M parameters
- **IndicBART**: AI4Bharat's Indian-language BART model

**Requirements**: GPU (16GB+ VRAM for mT5-base; mT5-small runs on 4GB)

**Google Colab**: Upload this notebook + splits/ folder to Colab, then run.

**Prerequisites**: Run Notebooks 01 and 02 first.

In [ ]:
# Install dependencies (run once, then restart kernel)
!pip install transformers datasets accelerate sentencepiece protobuf sacrebleu torch -q

In [ ]:
import sys, json, os
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from pathlib import Path
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer, AutoModelForSeq2SeqLM,
    Seq2SeqTrainer, Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq, EarlyStoppingCallback,
)

sys.path.insert(0, str(Path('.').resolve()))
from utils.evaluation import compute_metrics, print_metrics

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
if DEVICE == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

Path('models').mkdir(exist_ok=True)
Path('figures').mkdir(exist_ok=True)

In [ ]:
# Load splits
def load_json(path):
    with open(path, encoding='utf-8') as f:
        return json.load(f)

train = load_json('splits/marathi_train.json')
val   = load_json('splits/marathi_val.json')
test  = load_json('splits/marathi_test.json')
print(f'Train: {len(train)}  Val: {len(val)}  Test: {len(test)}')

## 1. Dataset Helper

In [ ]:
PREFIX = 'generate equation: '
MAX_SRC_LEN = 128
MAX_TGT_LEN = 64


class MWPDataset(Dataset):
    def __init__(self, data, tokenizer, prefix=PREFIX):
        self.data = data
        self.tok  = tokenizer
        self.pfx  = prefix

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        src  = self.pfx + item['Problem']
        tgt  = item['Equation']

        model_inputs = self.tok(
            src, max_length=MAX_SRC_LEN, padding=False, truncation=True)
        with self.tok.as_target_tokenizer():
            labels = self.tok(
                tgt, max_length=MAX_TGT_LEN, padding=False, truncation=True)

        model_inputs['labels'] = labels['input_ids']
        return model_inputs


def build_hf_dataset(data, tokenizer):
    """Returns HuggingFace-compatible list of dicts."""
    from datasets import Dataset as HFDataset
    src_texts = [PREFIX + x['Problem']  for x in data]
    tgt_texts = [x['Equation'] for x in data]

    enc = tokenizer(
        src_texts, text_target=tgt_texts,
        max_length=MAX_SRC_LEN, max_target_length=MAX_TGT_LEN,
        padding=False, truncation=True,
    )
    return HFDataset.from_dict(enc)

## 2. mT5-small Fine-Tuning

In [ ]:
MT5_MODEL = 'google/mt5-small'  # ~300MB; swap to 'google/mt5-base' (~580MB) for better accuracy

mt5_tokenizer = AutoTokenizer.from_pretrained(MT5_MODEL)
mt5_model     = AutoModelForSeq2SeqLM.from_pretrained(MT5_MODEL, tie_word_embeddings=False)
print(f'mT5 parameters: {mt5_model.num_parameters():,}')

In [ ]:
from datasets import Dataset as HFDataset


def make_hf_ds(data, tokenizer, prefix=PREFIX):
    src_texts = [prefix + x['Problem']  for x in data]
    tgt_texts = [x['Equation'] for x in data]

    # Tokenize source and target separately — avoids text_target label-masking
    # bug in transformers >=4.40 with DataCollatorForSeq2Seq
    src_enc = tokenizer(src_texts, max_length=MAX_SRC_LEN,
                        truncation=True, padding=False)
    tgt_enc = tokenizer(tgt_texts, max_length=MAX_TGT_LEN,
                        truncation=True, padding=False)

    ds = HFDataset.from_dict({
        'input_ids'     : src_enc['input_ids'],
        'attention_mask': src_enc['attention_mask'],
        'labels'        : tgt_enc['input_ids'],
    })

    # Verify labels are non-empty
    n_valid = sum(1 for lab in ds['labels'] if len(lab) > 0)
    print(f'  Dataset: {len(ds)} samples, {n_valid} with non-empty labels')
    return ds


mt5_train_ds = make_hf_ds(train, mt5_tokenizer)
mt5_val_ds   = make_hf_ds(val,   mt5_tokenizer)

# Sanity check — print a decoded example
sample_labels = mt5_train_ds[0]['labels']
print(f'  Sample label tokens : {sample_labels}')
print(f'  Decoded             : {mt5_tokenizer.decode(sample_labels, skip_special_tokens=True)}')

In [ ]:
MT5_OUTPUT = 'models/mt5_marathi'

mt5_model = AutoModelForSeq2SeqLM.from_pretrained(MT5_MODEL, tie_word_embeddings=False)
print(f'mT5 parameters         : {mt5_model.num_parameters():,}')
print(f'decoder_start_token_id : {mt5_model.config.decoder_start_token_id}')

# ── Sanity check ─────────────────────────────────────────────────────────────
mt5_model.eval()
_s = mt5_train_ds[:2]
_iids = torch.nn.utils.rnn.pad_sequence(
    [torch.tensor(s) for s in _s['input_ids']], batch_first=True,
    padding_value=mt5_tokenizer.pad_token_id)
_amsk = (_iids != mt5_tokenizer.pad_token_id).long()
_labs = torch.nn.utils.rnn.pad_sequence(
    [torch.tensor(s) for s in _s['labels']], batch_first=True, padding_value=-100)
with torch.no_grad():
    _chk = mt5_model(input_ids=_iids, attention_mask=_amsk, labels=_labs)
print(f'Sanity loss: {_chk.loss.item():.4f}')
assert not torch.isnan(_chk.loss), 'NaN on fresh model'
del _s, _iids, _amsk, _labs, _chk

# ── Collator ─────────────────────────────────────────────────────────────────
_pad_id = mt5_tokenizer.pad_token_id


def mt5_collator(features):
    iids = torch.nn.utils.rnn.pad_sequence(
        [torch.tensor(f['input_ids'],  dtype=torch.long) for f in features],
        batch_first=True, padding_value=_pad_id)
    labs = torch.nn.utils.rnn.pad_sequence(
        [torch.tensor(f['labels'], dtype=torch.long) for f in features],
        batch_first=True, padding_value=-100)
    return {'input_ids': iids, 'attention_mask': (iids != _pad_id).long(), 'labels': labs}


# ── Manual training loop ──────────────────────────────────────────────────────
# Seq2SeqTrainer with predict_with_generate=True skips computing eval loss
# (returns None → displayed as NaN). A plain loop is identical to the sanity
# check and gives full visibility into every batch.
from torch.utils.data import DataLoader as TorchDataLoader
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup

EPOCHS, BATCH_SIZE, LR, PATIENCE = 20, 16, 5e-4, 4

train_dl = TorchDataLoader(mt5_train_ds, batch_size=BATCH_SIZE,
                           shuffle=True,  collate_fn=mt5_collator)
val_dl   = TorchDataLoader(mt5_val_ds,   batch_size=32,
                           shuffle=False, collate_fn=mt5_collator)

optimizer  = AdamW(mt5_model.parameters(), lr=LR, weight_decay=0.01)
scheduler  = get_linear_schedule_with_warmup(
    optimizer, num_warmup_steps=100,
    num_training_steps=len(train_dl) * EPOCHS)

mt5_model.to(DEVICE)

# Confirm first training batch loss before entering the loop
mt5_model.eval()
_b0 = next(iter(train_dl))
_b0 = {k: v.to(DEVICE) for k, v in _b0.items()}
print(f'\nFirst training batch — input_ids: {_b0["input_ids"].shape}, '
      f'labels: {_b0["labels"].shape}')
print(f'Valid label tokens (non -100): '
      f'{(_b0["labels"][0] != -100).sum().item()} / {_b0["labels"].shape[1]}')
with torch.no_grad():
    _loss0 = mt5_model(**_b0).loss
print(f'First-batch loss: {_loss0.item():.4f}')
del _b0, _loss0
mt5_model.train()

best_val, patience_cnt = float('inf'), 0

for epoch in range(1, EPOCHS + 1):
    # Train
    mt5_model.train()
    tr_loss, n_nan_tr = 0.0, 0
    for batch in train_dl:
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        loss = mt5_model(**batch).loss
        if torch.isnan(loss):
            n_nan_tr += 1
            continue
        loss.backward()
        torch.nn.utils.clip_grad_norm_(mt5_model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()
        tr_loss += loss.item()
    tr_loss /= max(len(train_dl) - n_nan_tr, 1)

    # Eval
    mt5_model.eval()
    val_loss, n_nan_val = 0.0, 0
    with torch.no_grad():
        for batch in val_dl:
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            l = mt5_model(**batch).loss
            if torch.isnan(l):
                n_nan_val += 1
            else:
                val_loss += l.item()
    val_loss /= max(len(val_dl) - n_nan_val, 1)

    nan_note = (f'  [NaN batches train={n_nan_tr} val={n_nan_val}]'
                if n_nan_tr or n_nan_val else '')
    print(f'Epoch {epoch:2d}/{EPOCHS} | Train: {tr_loss:.4f} | Val: {val_loss:.4f}{nan_note}')

    if val_loss < best_val:
        best_val = val_loss
        patience_cnt = 0
        mt5_model.save_pretrained(MT5_OUTPUT)
        mt5_tokenizer.save_pretrained(MT5_OUTPUT)
        print(f'  Saved best (val={best_val:.4f})')
    else:
        patience_cnt += 1
        if patience_cnt >= PATIENCE:
            print(f'Early stopping at epoch {epoch}')
            break

mt5_model = AutoModelForSeq2SeqLM.from_pretrained(
    MT5_OUTPUT, tie_word_embeddings=False)
print(f'\nDone. Best val loss: {best_val:.4f}')

In [ ]:
def generate_predictions(model, tokenizer, data, batch_size=32, prefix=PREFIX):
    model.eval()
    preds = []
    for i in range(0, len(data), batch_size):
        batch = data[i:i+batch_size]
        texts = [prefix + x['Problem'] for x in batch]
        enc   = tokenizer(texts, return_tensors='pt', padding=True,
                          truncation=True, max_length=MAX_SRC_LEN).to(DEVICE)
        with torch.no_grad():
            ids = model.generate(**enc, max_new_tokens=MAX_TGT_LEN, num_beams=4)
        decoded = tokenizer.batch_decode(ids, skip_special_tokens=True)
        preds.extend(decoded)
    return preds


mt5_preds = generate_predictions(mt5_model.to(DEVICE), mt5_tokenizer, test)
gold_test = [x['Equation'] for x in test]

# ── Debug: inspect raw predictions ──────────────────────────────────────────
print('Sample predictions vs gold (first 5):')
print('-' * 70)
for i in range(5):
    print(f'  Gold : {gold_test[i]}')
    print(f'  Pred : {repr(mt5_preds[i])}')
    print()

non_empty = sum(1 for p in mt5_preds if p.strip())
has_x     = sum(1 for p in mt5_preds if 'X' in p or 'x' in p)
has_eq    = sum(1 for p in mt5_preds if '=' in p)
print(f'Non-empty preds : {non_empty}/{len(mt5_preds)}')
print(f'Contains "X"    : {has_x}/{len(mt5_preds)}')
print(f'Contains "="    : {has_eq}/{len(mt5_preds)}')
print()

mt5_metrics = compute_metrics(gold_test, mt5_preds)
print_metrics('mT5-small (Monolingual Marathi)', mt5_metrics)

## 3. IndicBART Fine-Tuning

In [ ]:
INDIC_MODEL = 'ai4bharat/IndicBART'

indic_tokenizer = AutoTokenizer.from_pretrained(INDIC_MODEL, do_lower_case=False, use_fast=False)
indic_model     = AutoModelForSeq2SeqLM.from_pretrained(INDIC_MODEL)
print(f'IndicBART parameters: {indic_model.num_parameters():,}')


def make_indic_ds(data, tokenizer, prefix='<2mr> '):
    src_texts = [prefix + x['Problem']  for x in data]
    tgt_texts = [x['Equation'] for x in data]

    src_enc = tokenizer(src_texts, max_length=MAX_SRC_LEN,
                        truncation=True, padding=False)
    tgt_enc = tokenizer(tgt_texts, max_length=MAX_TGT_LEN,
                        truncation=True, padding=False)

    ds = HFDataset.from_dict({
        'input_ids'     : src_enc['input_ids'],
        'attention_mask': src_enc['attention_mask'],
        'labels'        : tgt_enc['input_ids'],
    })

    n_valid = sum(1 for lab in ds['labels'] if len(lab) > 0)
    print(f'  Dataset: {len(ds)} samples, {n_valid} with non-empty labels')
    return ds


indic_train_ds = make_indic_ds(train, indic_tokenizer)
indic_val_ds   = make_indic_ds(val,   indic_tokenizer)

# Sanity check
sample_labels = indic_train_ds[0]['labels']
print(f'  Sample label tokens : {sample_labels}')
print(f'  Decoded             : {indic_tokenizer.decode(sample_labels, skip_special_tokens=True)}')

In [ ]:
INDIC_OUTPUT  = 'models/indicbart_marathi'
_indic_pad_id = indic_tokenizer.pad_token_id

indic_model = AutoModelForSeq2SeqLM.from_pretrained(INDIC_MODEL)
print(f'IndicBART parameters         : {indic_model.num_parameters():,}')
print(f'decoder_start_token_id       : {indic_model.config.decoder_start_token_id}')


def indic_collator(features):
    iids = torch.nn.utils.rnn.pad_sequence(
        [torch.tensor(f['input_ids'], dtype=torch.long) for f in features],
        batch_first=True, padding_value=_indic_pad_id)
    labs = torch.nn.utils.rnn.pad_sequence(
        [torch.tensor(f['labels'], dtype=torch.long) for f in features],
        batch_first=True, padding_value=-100)
    return {'input_ids': iids, 'attention_mask': (iids != _indic_pad_id).long(),
            'labels': labs}


EPOCHS, BATCH_SIZE, LR, PATIENCE = 20, 16, 5e-4, 4

indic_train_dl = TorchDataLoader(indic_train_ds, batch_size=BATCH_SIZE,
                                  shuffle=True,  collate_fn=indic_collator)
indic_val_dl   = TorchDataLoader(indic_val_ds,   batch_size=32,
                                  shuffle=False, collate_fn=indic_collator)

indic_optimizer = AdamW(indic_model.parameters(), lr=LR, weight_decay=0.01)
indic_scheduler = get_linear_schedule_with_warmup(
    indic_optimizer, num_warmup_steps=100,
    num_training_steps=len(indic_train_dl) * EPOCHS)

indic_model.to(DEVICE)
best_val_indic, patience_cnt_indic = float('inf'), 0

for epoch in range(1, EPOCHS + 1):
    indic_model.train()
    tr_loss, n_nan_tr = 0.0, 0
    for batch in indic_train_dl:
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        loss = indic_model(**batch).loss
        if torch.isnan(loss):
            n_nan_tr += 1
            continue
        loss.backward()
        torch.nn.utils.clip_grad_norm_(indic_model.parameters(), 1.0)
        indic_optimizer.step()
        indic_scheduler.step()
        indic_optimizer.zero_grad()
        tr_loss += loss.item()
    tr_loss /= max(len(indic_train_dl) - n_nan_tr, 1)

    indic_model.eval()
    val_loss, n_nan_val = 0.0, 0
    with torch.no_grad():
        for batch in indic_val_dl:
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            l = indic_model(**batch).loss
            if torch.isnan(l):
                n_nan_val += 1
            else:
                val_loss += l.item()
    val_loss /= max(len(indic_val_dl) - n_nan_val, 1)

    nan_note = (f'  [NaN batches train={n_nan_tr} val={n_nan_val}]'
                if n_nan_tr or n_nan_val else '')
    print(f'Epoch {epoch:2d}/{EPOCHS} | Train: {tr_loss:.4f} | Val: {val_loss:.4f}{nan_note}')

    if val_loss < best_val_indic:
        best_val_indic = val_loss
        patience_cnt_indic = 0
        indic_model.save_pretrained(INDIC_OUTPUT)
        indic_tokenizer.save_pretrained(INDIC_OUTPUT)
        print(f'  Saved best (val={best_val_indic:.4f})')
    else:
        patience_cnt_indic += 1
        if patience_cnt_indic >= PATIENCE:
            print(f'Early stopping at epoch {epoch}')
            break

indic_model = AutoModelForSeq2SeqLM.from_pretrained(INDIC_OUTPUT)
print(f'\nDone. Best val loss: {best_val_indic:.4f}')

In [ ]:
indic_preds   = generate_predictions(indic_model.to(DEVICE), indic_tokenizer,
                                     test, prefix='<2mr> ')
indic_metrics = compute_metrics(gold_test, indic_preds)
print_metrics('IndicBART (Monolingual Marathi)', indic_metrics)

## 4. Results: Transformer vs Baselines

In [ ]:
# Load baseline results if available
baseline_results = {}
try:
    prev = pd.read_csv('figures/table_5_1_baseline_results.csv')
    for _, row in prev.iterrows():
        baseline_results[row['Model']] = {
            'answer_accuracy'   : float(row['Answer Acc (%)']),
            'equation_accuracy' : float(row['Equation Acc (%)']),
            'bleu'              : float(row['BLEU']),
        }
except FileNotFoundError:
    print('Baseline results file not found — run Notebook 02 first')

all_results = {
    **baseline_results,
    'mT5-small'  : mt5_metrics,
    'IndicBART'  : indic_metrics,
}

df_all = pd.DataFrame([
    {
        'Model'             : name,
        'Answer Acc (%)'    : f"{m['answer_accuracy']:.1f}",
        'Equation Acc (%)'  : f"{m['equation_accuracy']:.1f}",
        'BLEU'              : f"{m['bleu']:.1f}",
    }
    for name, m in all_results.items()
])

print('\nAll Models Results (Test Set — Monolingual Marathi)')
print(df_all.to_string(index=False))
df_all.to_csv('figures/table_5_2_transformer_results.csv', index=False)

In [ ]:
# Save predictions for error analysis (Notebook 05)
preds_out = [
    {
        'pIndex'   : item['pIndex'],
        'Problem'  : item['Problem'],
        'Gold'     : item['Equation'],
        'mT5'      : mt5_preds[i],
        'IndicBART': indic_preds[i],
    }
    for i, item in enumerate(test)
]
with open('splits/test_predictions_transformers.json', 'w', encoding='utf-8') as f:
    json.dump(preds_out, f, ensure_ascii=False, indent=2)
print('Saved: splits/test_predictions_transformers.json')

## 5. Performance by Operator Count

In [ ]:
from utils.evaluation import answers_match

for model_name, preds in [('mT5-small', mt5_preds), ('IndicBART', indic_preds)]:
    print(f'\n{model_name} — Answer Accuracy by Operator Count:')
    for n_ops in sorted(set(x['Number of Operators'] for x in test)):
        subset_gold = [x['Equation'] for x in test if x['Number of Operators'] == n_ops]
        subset_pred = [preds[i] for i, x in enumerate(test) if x['Number of Operators'] == n_ops]
        acc = sum(answers_match(g, p) for g, p in zip(subset_gold, subset_pred))
        print(f'  {n_ops} operator(s): {acc}/{len(subset_gold)} = {acc/len(subset_gold)*100:.1f}%')